# Форма и материал почти закрытой фигуры

Добавьте Competition Data и приватный clevr_curriculum. Настройте GPU T4 ×2 и выключите Интернет. Прогон сравнивает 18 конфигураций по двум seed. Результаты сохраняются и при падении отдельной ветки.

In [ ]:
import hashlib
import json
import os
from pathlib import Path
import shutil
import subprocess
import sys
import tempfile
import time
import zipfile
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import pandas as pd
import torch
from IPython.display import FileLink, display
from scipy.special import expit
from sklearn.metrics import roc_auc_score

INPUT = Path("/kaggle/input")
WORK = Path("/kaggle/working")
assert torch.cuda.device_count() == 2, "Выберите GPU T4 x2"
ROOT = Path(tempfile.mkdtemp(prefix="clevr_verify_"))
RESULT = ROOT / "verify_result"
RESULT.mkdir()
print("Рабочая папка", ROOT, flush=True)

def unpack(path, dest):
    with zipfile.ZipFile(path) as z:
        for name in z.namelist():
            if not (dest / name).resolve().is_relative_to(dest.resolve()):
                raise ValueError(name)
        z.extractall(dest)

def find_data(parent):
    return [p.parent for p in parent.rglob("train.csv")
            if all((p.parent / n).is_file() for n in ("val.csv","test.csv","sample_submission.csv"))
            and all((p.parent / n).is_dir() for n in ("train","val","test"))]

locations = list(INPUT.rglob("curriculum_manifest.json"))
if not locations:
    archives = list(INPUT.rglob("clevr_curriculum.zip"))
    if len(archives) != 1:
        raise ValueError("Подключите один Private Dataset из clevr_curriculum.zip")
    unpack(archives[0], ROOT / "package")
    locations = [ROOT / "package/curriculum_manifest.json"]
if len(locations) != 1:
    raise ValueError("Подключено несколько версий весов")
package = locations[0].parent
for key, digest in json.loads(locations[0].read_text()).items():
    with (package / key).open("rb") as f:
        if hashlib.file_digest(f, "sha256").hexdigest() != digest:
            raise ValueError(key)
datasets = find_data(INPUT)
if not datasets:
    for j, path in enumerate(INPUT.rglob("*.zip")):
        with zipfile.ZipFile(path) as z:
            names = {Path(k).name for k in z.namelist()}
        if {"train.csv","val.csv","test.csv","sample_submission.csv"} <= names:
            unpack(path, ROOT / "inputs" / str(j))
    datasets = find_data(ROOT / "inputs")
if len(datasets) != 1:
    raise ValueError("Подключите данные соревнования")
DATA = datasets[0]
env = dict(os.environ, CLEVR_DATA=str(DATA), CLEVR_PRIORS=str(package/"priors"),
           CLEVR_WEIGHTS=str(package/"weights_in"), PYTHONPATH=str(package),
           PYTHONUNBUFFERED="1", OMP_NUM_THREADS="2", OPENBLAS_NUM_THREADS="2", MKL_NUM_THREADS="2")

In [ ]:
MODES = (('unet', 'legacy'), ('r50', 'legacy'), ('unet', 'mixed'), ('unet', 'staged'), ('unet', 'noexact'), ('unet', 'boundary'), ('r50', 'mixed'), ('r50', 'staged'), ('r50', 'noexact'), ('efficient', 'mixed'), ('efficient', 'staged'), ('r18', 'mixed'), ('r18', 'staged'), ('convnext', 'mixed'), ('convnext', 'staged'), ('r101', 'staged'), ('scene', 'staged'), ('scene', 'noexact'))

assert all("T4" in torch.cuda.get_device_name(i) for i in range(2)), "Выберите GPU T4 ×2"
env.update(FRAGMENT_INPUT=str(package), OMP_NUM_THREADS="2", MKL_NUM_THREADS="2", OPENBLAS_NUM_THREADS="2", NUMEXPR_NUM_THREADS="2")

RESULT = ROOT/"curriculum"
RESULT.mkdir(exist_ok=True)
ARMS = ("rgb","edge","unet","r50","r50native","r50real","r18","convnext","efficient")
SEEDS = (421,853)
EPOCHS=24
SCENE_EPOCHS=24
V="scripts.research.rim_fragments"
U="scripts.research.rim_scene_unet"
R="scripts.research.rear_candidate"
E="scripts.research.rim_eval"
from threading import Lock
failure_lock=Lock()
failures=[]
def launch(gpu, module, *args):
    step = module.rsplit(".",1)[-1]+"_"+"_".join(map(str,args)).replace("--","")
    done = RESULT / f"{step}.done"
    if done.exists():
        return
    log = RESULT / f"{step}.log"
    cmd = [sys.executable, "-m", module, *map(str,args)]
    with log.open("a", buffering=1) as out, subprocess.Popen(cmd, cwd=ROOT, text=True,
            env={**env,"CUDA_VISIBLE_DEVICES":str(gpu)}, stdout=subprocess.PIPE, stderr=subprocess.STDOUT) as job:
        for line in job.stdout:
            print(f"[GPU {gpu}] {line}", end="", flush=True)
            out.write(line)
        if job.wait():
            shutil.copy2(log, WORK/f"curriculum_{step}_failed.log")
            raise RuntimeError(f"{step} завершился с ошибкой. Лог сохранён в Output.")
    done.touch()

def parallel(jobs):
    with ThreadPoolExecutor(max_workers=2) as pool:
        results = [pool.submit(fn,i,*args) for i,(fn,args) in enumerate(jobs)]
        for result in results:
            result.result()

def monitor(job, stage):
    telemetry = RESULT / f"gpu_{stage}.csv"
    with telemetry.open("w",buffering=1) as out:
        query = ["nvidia-smi", "--query-gpu=timestamp,index,utilization.gpu,utilization.memory,memory.used,temperature.gpu",
                 "--format=csv", "-l", "2"]
        try:
            process = subprocess.Popen(query,stdout=out,stderr=subprocess.STDOUT)
        except FileNotFoundError:
            process = None
        try:
            return job()
        finally:
            if process is not None:
                process.terminate()
                process.wait()

def safe(gpu,module,*args):
    try:
        launch(gpu,module,*args)
        return True
    except Exception as exc:
        record=dict(job=module,args=list(args),error=str(exc))
        with failure_lock:
            failures.append(record)
            (RESULT/"failures.json").write_text(json.dumps(failures,indent=2))
        print("Сбой шага",record,flush=True)
        return False

def pair(module, tasks):
    with ThreadPoolExecutor(2) as pool:
        pending=[pool.submit(safe,gpu,module,*args) for gpu,args in enumerate(tasks)]
        return [job.result() for job in pending]


from collections import defaultdict
import torchvision
MODEL="scripts.research.curriculum_train"
REPORT="scripts.research.curriculum_eval"
OUTPUT=WORK/"fragment_checkpoints"
OUTPUT.mkdir(exist_ok=True)
RUN_STEPS=1200
(RESULT/"runtime.json").write_text(json.dumps(dict(torch=torch.__version__,torchvision=torchvision.__version__,
    cuda=torch.version.cuda,gpu=[torch.cuda.get_device_name(i) for i in range(2)],steps=RUN_STEPS)))

def preserve(arm,mode):
    tag=f"{arm}_{mode}"
    for p in RESULT.glob(f"{tag}_*"):
        if p.suffix in (".pt",".json",".npz") and not (OUTPUT/p.name).exists():
            shutil.copy2(p,OUTPUT/p.name)

def extraction(split):
    (ROOT/"rim").mkdir(exist_ok=True)
    if not all(pair(V,[("prepare","--split",split,"--shard",k) for k in (0,1)])):
        return False
    for k in (0,1):
        for extension in (".npz","_rgb.npy"):
            src=ROOT/"rim"/f"{split}_{k}{extension}"
            dst=RESULT/src.name
            if src.exists():
                src.replace(dst)
            elif not dst.exists():
                return False
    return True

def posterior(split):
    return all(pair(R,[("prepare","--split",split,"--shard",k) for k in (0,1)]))

def jobs(arm,mode,stage,*args):
    return [(stage,"--arm",arm,"--mode",mode,"--seed",s,*args) for s in SEEDS]

def inferred(arm,mode,split,ready=None,extra=()):
    if ready is None:
        ready=[True,True]
    if arm=="scene":
        tasks=jobs(arm,mode,"infer","--split",split)
        if all(ready):
            return all(pair(MODEL,tasks))
        for gpu,ok in enumerate(ready):
            if ok:
                safe(gpu,MODEL,*tasks[gpu])
        return False
    result=True
    for seed,ok in zip(SEEDS,ready):
        if ok:
            tasks=[("infer","--arm",arm,"--mode",mode,"--seed",seed,
                    "--split",split,"--shard",shard,*extra) for shard in (0,1)]
            result=all(pair(MODEL,tasks)) and result
    return result and all(ready)

def run_batch_probes():
    mapping={}
    for arm,mode in MODES:
        kind="legacy" if mode=="legacy" else "staged"
        if (arm,kind) in mapping:
            continue
        if all(pair(MODEL,jobs(arm,kind,"probe"))):
            profiles=[json.loads((RESULT/f"probe_{arm}_{kind}_{s}.json").read_text()) for s in SEEDS]
            mapping[arm,kind]=min(o["batch"] for o in profiles)
    # Keep the same microbatch and total number of training draws in the depth ablation.
    if ("r50","staged") in mapping and ("r101","staged") in mapping:
        common=min(mapping["r50","staged"],mapping["r101","staged"])
        mapping["r50","staged"]=mapping["r101","staged"]=common
    print("Measured batches",mapping,flush=True)
    return mapping

def train():
    prepared=extraction("val")
    mapping=run_batch_probes()
    for arm,mode in MODES:
        kind="legacy" if mode=="legacy" else "staged"
        batch=mapping.get((arm,kind))
        if not batch:
            continue
        trained=pair(MODEL,jobs(arm,mode,"fit","--batch",batch,"--steps",RUN_STEPS))
        # A failed late checkpoint may still have a usable earlier selected state.
        ready=[(RESULT/f"{arm}_{mode}_{s}.pt").exists() for s in SEEDS]
        preserve(arm,mode)
        if prepared or arm=="scene":
            inferred(arm,mode,"val",ready)
            if mode=="legacy":
                for checkpoint in (240,1200):
                    if checkpoint<=RUN_STEPS:
                        inner_ready=[(RESULT/f"{arm}_{mode}_{s}_{checkpoint}.pt").exists() for s in SEEDS]
                        inferred(arm,mode,"val",inner_ready,("--checkpoint",checkpoint))
                        preserve(arm,f"{mode}{checkpoint}")
        preserve(arm,mode)
    if prepared:
        posterior("val")

monitor(train,"train")

## Исходные сцены val

Выбор контрольных точек происходит на отдельных изображениях train. Правила ансамбля сравниваются на фолдах из целых сцен. Ручные четыре области открываются для диагностики после выбора.

In [ ]:
complete=safe(0,REPORT)
decisions=json.loads((RESULT/"decisions.json").read_text()) if complete else {}
if decisions:
    print("Base ROC-AUC",decisions["base"]["auc"])
    for seed,item in decisions["folds"].items():
        print("Grouped folds",seed,item["auc"],item["auc_excluding_diagnostic"],item["delta_ci95"])
    print("Prospective",decisions["prospective"],"Selected",decisions["selected"])

In [ ]:
def requirements(plan):
    kind,*args=plan
    if kind=="base":
        return set()
    if kind=="scene":
        return {args[0]}
    if kind=="joint":
        return requirements(args[0])|requirements(args[1])
    return set(args[0])

def test():
    plan=decisions.get("selected")
    if not plan:
        return
    required=requirements(plan)
    local=[n for n in sorted(required) if not n.startswith("scene_")]
    if local and (not extraction("test") or not posterior("test")):
        return
    for name in sorted(required):
        arm,mode=name.split("_",1)
        skip=("--checkpoint",int(mode.removeprefix("legacy"))) if mode in ("legacy240","legacy1200") else ()
        if skip:
            mode="legacy"
        if not inferred(arm,mode,"test",extra=skip):
            return
    name="visible_unet" if plan[0]=="scene" else "visible_mix" if len(required)>1 else next(iter(required))
    if safe(0,REPORT,"--export","--filename",name):
        shutil.copy2(RESULT/f"{name}.csv",WORK/f"{name}.csv")
        display(FileLink(str(WORK/f"{name}.csv")))

try:
    monitor(test,"test")
finally:
    archive=WORK/"curriculum_results.zip"
    with zipfile.ZipFile(archive,"w",compression=zipfile.ZIP_DEFLATED,compresslevel=1) as z:
        for path in sorted(RESULT.iterdir()):
            if path.is_file() and path.suffix in (".json",".npz",".pt",".log",".csv"):
                # Selected states duplicate the versioned checkpoints and can be reconstructed from JSON.
                if path.suffix==".pt" and len(path.stem.rsplit("_",2))==3 and not path.stem.rsplit("_",2)[-2].isdigit():
                    continue
                z.write(path,path.name)
    display(FileLink(str(archive)))